# From Post To Personality (P2P): MBTI Prediction from Social Media

**Paper:** Ma et al., *From Post To Personality: Harnessing LLMs for MBTI Prediction in Social Media*, CIKM 2025

This notebook reproduces the P2P framework — a dual-LLM pipeline that predicts Myers-Briggs personality types from social media posts using:
- A **fine-tuned local LLM** (DeepSeek-R1-8B with QLoRA) for personality feature extraction
- **Retrieval-Augmented Generation (RAG)** with FAISS to find similar labeled users as in-context demonstrations
- An **online LLM** (DeepSeek-V3) for the final MBTI prediction

### Pipeline Overview

```
User Posts → [Fine-tuned Local LLM] → Personality Features (h)
                                              │
                                              ├── Sentence-BERT embedding (768-dim)
                                              ├── LLM hidden states (4096-dim)
                                              └── Concatenate → FAISS query
                                                        │
                                                        ▼
                                                  k=5 similar users
                                                        │
                  Posts + Features + Similar Users ──────┘
                                    │
                                    ▼
                          [Online LLM (DeepSeek-V3)]
                                    │
                                    ▼
                              4-letter MBTI type
```

### Table of Contents

1. **Environment Setup** — Dependencies, GPU check, configuration
2. **Data Loading** — Load preprocessed PersonalityCafe dataset (8,675 users)
3. **Fine-Tuning the Local LLM** — QLoRA + SMOTE for class imbalance (Paper §3.2)
4. **Personality Feature Extraction** — Generate textual assessments per user (Paper §3.1, Appendix C)
5. **Building the RAG Vector Database** — Sentence-BERT + LLM hidden states → FAISS index (Paper §3.1)
6. **Online MBTI Prediction** — RAG-augmented prediction via DeepSeek-V3 API (Paper §3.1, Appendix C)
7. **Evaluation** — Accuracy, F1, AUC per dimension + baseline comparisons (Paper §4.1, Table 1)

---
## 1. Environment Setup

In [ ]:
# 1.1 Install dependencies
!pip install -qqq --upgrade --no-cache-dir \
    "unsloth" "unsloth_zoo" \
    "huggingface_hub>=0.28" \
    "transformers>=4.51.3,<=4.57.6" \
    "trl>=0.18.2,<=0.24.0" \
    "peft>=0.18.0" \
    "accelerate>=0.34.1" \
    "bitsandbytes>=0.44" \
    "datasets>=3.0,<4.4" \
    "sentence-transformers>=3.0" \
    "faiss-cpu>=1.8" \
    "scikit-learn>=1.5" \
    "xgboost>=2.1" \
    "openai>=1.0" \
    "python-dotenv>=1.0" \
    "pandas>=2.2" "pyarrow>=17" \
    "matplotlib>=3.9" "seaborn>=0.13"

In [ ]:
# 1.2 Imports and GPU verification
import unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch, os, time, re, json, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} | {round(p.total_memory/1e9,1)} GB')
print('BF16 supported:', is_bfloat16_supported())
assert torch.cuda.is_available(), 'GPU required for fine-tuning and inference.'

In [ ]:
# 1.3 Configuration and hyperparameters (Paper §4.1)
WORK_DIR = Path('.').resolve()
ARTIFACTS = WORK_DIR / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)

# Preprocessed data
SPLIT_PARQUET = ARTIFACTS / 'split.parquet'
assert SPLIT_PARQUET.exists(), f'split.parquet not found at {SPLIT_PARQUET}'

# DeepSeek API key for online prediction
from dotenv import load_dotenv
load_dotenv(WORK_DIR / '.env')
api_key = os.environ.get('DEEPSEEK_API_KEY', '')
assert api_key and api_key != 'sk-paste-your-key-here', 'Set DEEPSEEK_API_KEY in .env'

# Model
BASE_MODEL       = 'unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit'
MAX_SEQ_LEN      = 1536

# LoRA hyperparameters (Paper §4.1)
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.0
LORA_TARGETS     = ['q_proj', 'v_proj']

# Training hyperparameters (Paper §4.1)
LEARNING_RATE    = 1e-4
BATCH_SIZE       = 4
GRAD_ACCUM       = 2
EPOCHS           = 3
MIN_SAMPLES      = 500
VAL_SUBSAMPLE    = 200

# RAG
K                = 5

print(f'Data: {SPLIT_PARQUET}')
print(f'API key: set')
print(f'Model: {BASE_MODEL}')

---
## 2. Data Loading

In [ ]:
# 2.1 Load preprocessed dataset (60/20/20 stratified split from Phase 0)
df_all = pd.read_parquet(SPLIT_PARQUET)
df_train = df_all[df_all['split'] == 'train'].reset_index(drop=True)
df_val   = df_all[df_all['split'] == 'val'].reset_index(drop=True)
df_test  = df_all[df_all['split'] == 'test'].reset_index(drop=True)

print(f'Total: {len(df_all)} users')
print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')
print(f'MBTI types: {df_train["type"].nunique()}')
print(f'Class imbalance: {df_train["type"].value_counts().max()} (max) / {df_train["type"].value_counts().min()} (min)')

---
## 3. Fine-Tuning the Local LLM (Paper §3.2)

Fine-tune DeepSeek-R1-Distill-Llama-8B using QLoRA (quantized low-rank adaptation) with SMOTE-style oversampling to address the severe class imbalance (INFP has 47x more samples than ESTJ).

The model learns to predict MBTI type directly from user posts, which improves its personality feature extraction in the next phase.

In [ ]:
# 3.1 SMOTE-style oversampling for minority MBTI classes
ADAPTER_PATH = ARTIFACTS / 'lora_adapter'

if ADAPTER_PATH.exists():
    print(f'Fine-tuned adapter found at {ADAPTER_PATH} — skipping training')
    SKIP_TRAINING = True
else:
    SKIP_TRAINING = False

    def shuffle_posts(text):
        """Shuffle word-chunks to create slightly varied duplicates for oversampling."""
        words = text.split()
        if len(words) < 50: return text
        chunks = [' '.join(words[i:i+10]) for i in range(0, len(words), 10)]
        random.shuffle(chunks)
        return ' '.join(chunks)

    # Oversample minority classes to MIN_SAMPLES each
    pieces = []
    for mbti, grp in df_train.groupby('type'):
        pieces.append(grp)
        need = max(0, MIN_SAMPLES - len(grp))
        for _ in range(need):
            row = grp.sample(1, random_state=random.randint(0, 999999)).iloc[0].copy()
            row['posts_clean'] = shuffle_posts(row['posts_clean'])
            pieces.append(pd.DataFrame([row]))

    df_train_bal = pd.concat(pieces, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f'Before oversampling: {len(df_train)} | After: {len(df_train_bal)}')
    print(df_train_bal['type'].value_counts().to_string())

In [ ]:
# 3.2 Format training data as Alpaca-style instructions
if not SKIP_TRAINING:
    from datasets import Dataset

    INSTRUCTION_TEMPLATE = (
        "Below is an instruction that describes a task. Write a response that "
        "appropriately completes the request.\n\n"
        "### Instruction:\n"
        "Predict the 4-letter Myers-Briggs personality type of the user from "
        "their social-media posts. Output must be exactly one of: INFP, INFJ, "
        "INTP, INTJ, ENFP, ENFJ, ENTP, ENTJ, ISFP, ISFJ, ISTP, ISTJ, ESFP, ESFJ, "
        "ESTP, ESTJ. Return only four uppercase letters.\n\n"
        "### Input:\n{posts}\n\n"
        "### Response:\n{mbti}"
    )

    def to_instruction(row):
        return {'text': INSTRUCTION_TEMPLATE.format(posts=row['posts_clean'], mbti=row['type'])}

    df_val_sub = df_val.sample(min(VAL_SUBSAMPLE, len(df_val)), random_state=SEED)
    train_ds = Dataset.from_pandas(df_train_bal[['posts_clean', 'type']]).map(to_instruction, remove_columns=['posts_clean', 'type'])
    val_ds = Dataset.from_pandas(df_val_sub[['posts_clean', 'type']]).map(to_instruction, remove_columns=['posts_clean', 'type'])
    print(f'Training samples: {len(train_ds)} | Validation samples: {len(val_ds)}')

In [ ]:
# 3.3 Load base model and attach LoRA adapters
if not SKIP_TRAINING:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)

    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS, bias='none',
        use_gradient_checkpointing='unsloth', random_state=SEED)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# 3.4 Train with SFTTrainer (AdamW, lr=1e-4, early stopping)
if not SKIP_TRAINING:
    from trl import SFTTrainer, SFTConfig
    from transformers import EarlyStoppingCallback

    sft_config = SFTConfig(
        output_dir=str(ARTIFACTS / 'checkpoints'),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim='adamw_8bit',
        lr_scheduler_type='linear',
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=500,
        save_strategy='steps',
        save_steps=500,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        seed=SEED,
        report_to='none',
        dataset_text_field='text',
        max_length=MAX_SEQ_LEN,
        packing=False,
    )

    trainer = SFTTrainer(
        model=model, processing_class=tokenizer,
        train_dataset=train_ds, eval_dataset=val_ds,
        args=sft_config,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

    print('Starting training...')
    stats = trainer.train()
    print(f'Training loss: {stats.metrics["train_loss"]:.4f} | Time: {stats.metrics["train_runtime"]/60:.1f} min')

    model.save_pretrained(str(ADAPTER_PATH))
    tokenizer.save_pretrained(str(ADAPTER_PATH))
    print(f'Adapter saved to {ADAPTER_PATH}')

---
## 4. Personality Feature Extraction (Paper §3.1, Appendix C)

For each user, the fine-tuned local LLM reads their posts and generates an interpretable personality assessment covering the four MBTI dimensions (E/I, S/N, T/F, J/P). These textual features serve as input to both the FAISS vector database and the final prediction prompt.

In [ ]:
# 4.1 Load the fine-tuned model for inference
FEATURES_PARQUET = ARTIFACTS / 'features_h.parquet'

if FEATURES_PARQUET.exists():
    print(f'Features cached at {FEATURES_PARQUET} — skipping model load')
else:
    if 'model' not in dir() or model is None:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)

    if ADAPTER_PATH.exists():
        from peft import PeftModel
        if not hasattr(model, 'peft_config'):
            model = PeftModel.from_pretrained(model, str(ADAPTER_PATH))
        print('Fine-tuned adapter loaded')
    else:
        print('No adapter found — using base model')

    FastLanguageModel.for_inference(model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# 4.2 Extract personality features for all users
#     Prompt template from paper Appendix C

FEATURE_PROMPT = (
    'According to the following content <CONTENT>, extract the key features '
    'from these perspectives:\n'
    '1. Social tendency (Extraversion E / Introversion I)\n'
    '2. Information processing mode (Sensing S / iNtuition N)\n'
    '3. Decision-making mode (Thinking T / Feeling F)\n'
    '4. Lifestyle (Judging J / Perceiving P)'
)

def extract_features_local(posts_clean, max_input_tokens=1200):
    tokens = tokenizer.encode(posts_clean, add_special_tokens=False)[:max_input_tokens]
    truncated = tokenizer.decode(tokens, skip_special_tokens=True)
    prompt = FEATURE_PROMPT.replace('<CONTENT>', truncated)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False, repetition_penalty=1.1)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

if FEATURES_PARQUET.exists():
    df_feat = pd.read_parquet(FEATURES_PARQUET)
    print(f'Loaded cached features: {len(df_feat)} users')
else:
    df_feat = df_all.copy()
    df_feat['features_h'] = ''
    t0 = time.time()
    for i in tqdm(range(len(df_feat)), desc='Extracting features', ncols=80):
        df_feat.at[i, 'features_h'] = extract_features_local(df_feat.iloc[i]['posts_clean'])
        if (i + 1) % 500 == 0:
            df_feat.to_parquet(FEATURES_PARQUET, index=False)
            elapsed = time.time() - t0
            eta = elapsed / (i + 1) * (len(df_feat) - i - 1)
            print(f'  Checkpoint: {i+1}/{len(df_feat)} | {elapsed/60:.1f}min | ETA: {eta/60:.1f}min')
    df_feat.to_parquet(FEATURES_PARQUET, index=False)
    print(f'Feature extraction complete: {(time.time()-t0)/60:.1f} min')

---
## 5. Building the RAG Vector Database (Paper §3.1)

Each training user is represented by a concatenation of:
- **Sentence-BERT embedding** (768-dim) of their original posts
- **LLM hidden state** (4096-dim) of their personality features from the fine-tuned model

These are indexed in FAISS for exact k-NN retrieval with L2 distance. At inference, the k=5 most similar training users (with known MBTI labels) serve as in-context demonstrations for the online LLM.

In [ ]:
# 5.1 Load embedding models
from sentence_transformers import SentenceTransformer
import faiss

FAISS_INDEX_PATH = ARTIFACTS / 'faiss.index'
FAISS_META_PATH  = ARTIFACTS / 'faiss_meta.parquet'
HS_TRAIN_PATH    = ARTIFACTS / 'hidden_states_train.npy'
HS_TEST_PATH     = ARTIFACTS / 'hidden_states_test.npy'

df_feat = pd.read_parquet(FEATURES_PARQUET)
df_train_feat = df_feat[df_feat['split'] == 'train'].reset_index(drop=True)
df_test_feat  = df_feat[df_feat['split'] == 'test'].reset_index(drop=True)

# Sentence-BERT: 768-dim (paper cites Reimers & Gurevych 2019)
sbert = SentenceTransformer('all-mpnet-base-v2')
print(f'Sentence-BERT: all-mpnet-base-v2 (dim={sbert.get_sentence_embedding_dimension()})')

# Ensure LLM is loaded for hidden state extraction
if 'model' not in dir() or model is None:
    from transformers import AutoModelForCausalLM, AutoTokenizer as AT
    tokenizer = AT.from_pretrained(BASE_MODEL, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, device_map='auto', trust_remote_code=True, torch_dtype=torch.float16)
    if ADAPTER_PATH.exists():
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(ADAPTER_PATH))
    model.eval()
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f'LLM loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# 5.2 Extract hidden states from the fine-tuned LLM's last layer

def extract_hidden_state(text, max_tokens=512):
    """Forward pass through LLM, return avg-pooled last hidden layer."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=max_tokens, padding=False).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    last_hidden = outputs.hidden_states[-1][0]  # (seq_len, hidden_dim)
    return last_hidden.mean(dim=0).cpu().float().numpy()

# Training set hidden states
if HS_TRAIN_PATH.exists():
    hs_train = np.load(HS_TRAIN_PATH)
    print(f'Loaded cached train hidden states: {hs_train.shape}')
else:
    hidden_dim = extract_hidden_state('test').shape[0]
    print(f'Hidden dimension: {hidden_dim}')
    hs_train = np.zeros((len(df_train_feat), hidden_dim), dtype=np.float32)
    t0 = time.time()
    for i in tqdm(range(len(df_train_feat)), desc='Train hidden states', ncols=80):
        hs_train[i] = extract_hidden_state(df_train_feat.iloc[i]['features_h'])
        if (i + 1) % 1000 == 0:
            print(f'  {i+1}/{len(df_train_feat)} | {(time.time()-t0)/60:.1f}min')
    np.save(HS_TRAIN_PATH, hs_train)
    print(f'Saved: {hs_train.shape} | {(time.time()-t0)/60:.1f} min')

# Test set hidden states
if HS_TEST_PATH.exists():
    hs_test = np.load(HS_TEST_PATH)
    print(f'Loaded cached test hidden states: {hs_test.shape}')
else:
    hidden_dim = hs_train.shape[1]
    hs_test = np.zeros((len(df_test_feat), hidden_dim), dtype=np.float32)
    t0 = time.time()
    for i in tqdm(range(len(df_test_feat)), desc='Test hidden states', ncols=80):
        hs_test[i] = extract_hidden_state(df_test_feat.iloc[i]['features_h'])
    np.save(HS_TEST_PATH, hs_test)
    print(f'Saved: {hs_test.shape} | {(time.time()-t0)/60:.1f} min')

In [ ]:
# 5.3 Build FAISS index: Sentence-BERT (768-dim) + LLM hidden states (4096-dim)

print('Encoding training posts via Sentence-BERT...')
sbert_train = sbert.encode(df_train_feat['posts_clean'].tolist(), show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(sbert_train)

hs_train_norm = hs_train.copy()
faiss.normalize_L2(hs_train_norm)

combined_train = np.concatenate([sbert_train, hs_train_norm], axis=1)
print(f'Combined vector: {combined_train.shape} (SBERT {sbert_train.shape[1]} + hidden {hs_train.shape[1]})')

# Build and save FAISS index
index = faiss.IndexFlatL2(combined_train.shape[1])
index.add(combined_train)
faiss.write_index(index, str(FAISS_INDEX_PATH))
df_train_feat[['type', 'posts_clean', 'features_h']].to_parquet(FAISS_META_PATH, index=False)
print(f'FAISS index: {index.ntotal} vectors, dim={index.d}')

# Encode test set and retrieve neighbors
print('\nEncoding test posts via Sentence-BERT...')
sbert_test = sbert.encode(df_test_feat['posts_clean'].tolist(), show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(sbert_test)
hs_test_norm = hs_test.copy()
faiss.normalize_L2(hs_test_norm)
combined_test = np.concatenate([sbert_test, hs_test_norm], axis=1)

D, I = index.search(combined_test, K)
print(f'Retrieved top-{K} neighbors for {len(df_test_feat)} test users')

# Preview retrieval quality
df_meta = pd.read_parquet(FAISS_META_PATH)
for i in range(3):
    neighbor_types = [df_meta.iloc[j]['type'] for j in I[i]]
    print(f'  Test user {i} ({df_test_feat.iloc[i]["type"]}): neighbors = {neighbor_types}')

---
## 6. Online MBTI Prediction with RAG (Paper §3.1, Appendix C)

For each test user, the pipeline:
1. Retrieves k=5 most similar training users from FAISS (with known MBTI labels)
2. Constructs a prompt combining the user's posts, personality features, and retrieved demonstrations
3. Calls DeepSeek-V3 (the paper's online LLM) to predict the 4-letter MBTI type

In [ ]:
# 6.1 Setup API client and prediction prompts
import asyncio
from openai import OpenAI, AsyncOpenAI

client = OpenAI(api_key=api_key, base_url='https://api.deepseek.com')
async_client = AsyncOpenAI(api_key=api_key, base_url='https://api.deepseek.com')
MAX_CONCURRENT = 15

# Prediction prompt from paper Appendix C
PREDICT_PROMPT = (
    'According to the following content <CONTENT>, combined with the key features <FEATURES> '
    'extracted by the local model, refer to the MBTI distribution of similar texts <SIM-TEXTS>, '
    'and predict the MBTI type from four dimensions (only output four letters). '
    'The emphases of the four dimensions are as follows:\n'
    '1. Social tendency (Extraversion E / Introversion I)\n'
    '2. Information processing mode (Sensing S / iNtuition N)\n'
    '3. Decision-making mode (Thinking T / Feeling F)\n'
    '4. Lifestyle (Judging J / Perceiving P)'
)

SYSTEM_PROMPT = (
    'You are an MBTI personality type classifier. '
    'You MUST respond with ONLY 4 uppercase letters representing the MBTI type. '
    'Valid types: INFP, INFJ, INTP, INTJ, ENFP, ENFJ, ENTP, ENTJ, '
    'ISFP, ISFJ, ISTP, ISTJ, ESFP, ESFJ, ESTP, ESTJ. '
    'No explanation, no punctuation, no other text. Just 4 letters.'
)

VALID_TYPES = {'INFP','INFJ','INTP','INTJ','ENFP','ENFJ','ENTP','ENTJ',
               'ISFP','ISFJ','ISTP','ISTJ','ESFP','ESFJ','ESTP','ESTJ'}

PREDICTIONS_PARQUET = ARTIFACTS / 'predictions.parquet'

def parse_mbti(raw):
    """Extract a valid 4-letter MBTI type from model response."""
    text = raw.strip().upper()
    if text in VALID_TYPES: return text
    match = re.search(r'\b([EI][SN][TF][JP])\b', text)
    if match: return match.group(1)
    match = re.search(r'[EI][SN][TF][JP]', text)
    if match: return match.group(0)
    return 'PARSE_FAIL'

print('DeepSeek-V3 API ready')

In [ ]:
# 6.2 Predict MBTI for all test users (parallel API calls)

async def predict_async(posts, features, neighbors_idx, semaphore):
    """Call DeepSeek-V3 API with RAG prompt for one test user."""
    async with semaphore:
        sim_texts = []
        for rank, idx in enumerate(neighbors_idx, 1):
            nb_row = df_meta.iloc[idx]
            sim_texts.append(f"{rank}. {nb_row['type']}: '{nb_row['posts_clean'][:200]}'")
        prompt = (PREDICT_PROMPT
            .replace('<CONTENT>', posts[:3000])
            .replace('<FEATURES>', features[:500])
            .replace('<SIM-TEXTS>', '\n'.join(sim_texts)))
        for attempt in range(3):
            try:
                r = await async_client.chat.completions.create(
                    model='deepseek-chat',
                    messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                              {'role': 'user', 'content': prompt}],
                    max_tokens=10, temperature=0)
                return parse_mbti(r.choices[0].message.content)
            except Exception as e:
                if attempt < 2: await asyncio.sleep(2 ** attempt)
                else: return 'ERROR'

# Check for cached predictions
if PREDICTIONS_PARQUET.exists():
    df_pred = pd.read_parquet(PREDICTIONS_PARQUET)
    valid_count = df_pred['pred'].isin(VALID_TYPES).sum()
    if valid_count == len(df_pred):
        print(f'Predictions cached: {valid_count} valid')
    else:
        PREDICTIONS_PARQUET.unlink()
        df_pred = None

if not PREDICTIONS_PARQUET.exists():
    df_pred = df_test_feat.copy()
    df_pred['pred'] = ''
    todo = df_pred.index.tolist()
    print(f'Predicting {len(todo)} test users (parallel, {MAX_CONCURRENT} concurrent)...')
    t0 = time.time()
    sem = asyncio.Semaphore(MAX_CONCURRENT)
    for bs in range(0, len(todo), 100):
        batch = todo[bs:bs + 100]
        tasks = [predict_async(df_pred.iloc[i]['posts_clean'], df_pred.iloc[i]['features_h'], I[i], sem) for i in batch]
        results = await asyncio.gather(*tasks)
        for i, pred in zip(batch, results):
            df_pred.at[i, 'pred'] = pred
        df_pred.to_parquet(PREDICTIONS_PARQUET, index=False)
        done = min(bs + 100, len(todo))
        elapsed = time.time() - t0
        print(f'  {done}/{len(todo)} | {elapsed/60:.1f}min | ETA: {(len(todo)-done)/(done/elapsed)/60:.1f}min')
    valid_count = df_pred['pred'].isin(VALID_TYPES).sum()
    print(f'Complete: {valid_count} valid | {(df_pred["pred"]=="ERROR").sum()} errors | {(time.time()-t0)/60:.1f}min')

---
## 7. Evaluation (Paper §4.1)

Per-dimension Accuracy, F1-score, and AUC on the 4 binary MBTI classification tasks (I/E, N/S, T/F, J/P), compared against traditional ML baselines and the paper's reported results.

In [ ]:
# 7.1 P2P results per dimension
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

df_pred = pd.read_parquet(PREDICTIONS_PARQUET)
valid = df_pred[df_pred['pred'].isin(VALID_TYPES)].copy()
print(f'Valid predictions: {len(valid)} / {len(df_pred)}')

valid['pred_ie'] = (valid['pred'].str[0] == 'I').astype(int)
valid['pred_ns'] = (valid['pred'].str[1] == 'N').astype(int)
valid['pred_tf'] = (valid['pred'].str[2] == 'F').astype(int)
valid['pred_jp'] = (valid['pred'].str[3] == 'P').astype(int)

dims = [('I/E', 'ie', 'pred_ie'), ('N/S', 'ns', 'pred_ns'),
        ('T/F', 'tf', 'pred_tf'), ('J/P', 'jp', 'pred_jp')]

paper_full = {'I/E': 0.932, 'N/S': 0.948, 'T/F': 0.931, 'J/P': 0.886}
results = {}

print(f"\n{'Dim':>5} {'Acc':>8} {'F1':>8} {'AUC':>8} {'Paper':>8}")
print('-' * 45)
for name, tc, pc in dims:
    y_t, y_p = valid[tc].values, valid[pc].values
    a = accuracy_score(y_t, y_p)
    f = f1_score(y_t, y_p)
    try: u = roc_auc_score(y_t, y_p)
    except: u = 0.0
    results[name] = {'acc': a, 'f1': f, 'auc': u}
    print(f'{name:>5} {a:>8.4f} {f:>8.4f} {u:>8.4f} {paper_full[name]:>8.4f}')
print('-' * 45)
print(f"  Avg {np.mean([v['acc'] for v in results.values()]):>8.4f} "
      f"{np.mean([v['f1'] for v in results.values()]):>8.4f} "
      f"{np.mean([v['auc'] for v in results.values()]):>8.4f} "
      f"{np.mean(list(paper_full.values())):>8.4f}")

json.dump(results, open(ARTIFACTS / 'metrics.json', 'w'), indent=2)
print(f'\nSaved to {ARTIFACTS / "metrics.json"}')

In [ ]:
# 7.2 Baseline comparison (Paper Table 1)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

tfidf = TfidfVectorizer(max_features=10000, min_df=2, max_df=0.95)
X_tr = tfidf.fit_transform(df_train_feat['posts_clean'])
X_te = tfidf.transform(df_test_feat['posts_clean'])

baselines = {
    'Naive Bayes': MultinomialNB(alpha=1.0),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    'SVM (RBF)': SVC(kernel='rbf', C=10, gamma=0.1, random_state=SEED),
    'XGBoost': XGBClassifier(eta=0.1, max_depth=6, subsample=0.8, random_state=SEED, eval_metric='logloss', verbosity=0),
}

bl_results = {}
for bn, clf in baselines.items():
    r = {}
    for dn, col in [('I/E', 'ie'), ('N/S', 'ns'), ('T/F', 'tf'), ('J/P', 'jp')]:
        c = type(clf)(**clf.get_params())
        c.fit(X_tr, df_train_feat[col].values)
        yp = c.predict(X_te)
        r[dn] = {'acc': accuracy_score(df_test_feat[col].values, yp),
                 'f1': f1_score(df_test_feat[col].values, yp)}
    bl_results[bn] = r

# Comparison table
print(f"\n{'Approach':>20} {'I/E':>8} {'N/S':>8} {'T/F':>8} {'J/P':>8} {'Avg':>8}")
print('-' * 60)
for bn, r in bl_results.items():
    accs = [r[d]['acc'] for d in ['I/E', 'N/S', 'T/F', 'J/P']]
    print(f'{bn:>20} {accs[0]:>8.4f} {accs[1]:>8.4f} {accs[2]:>8.4f} {accs[3]:>8.4f} {np.mean(accs):>8.4f}')
p2p_accs = [results[d]['acc'] for d in ['I/E', 'N/S', 'T/F', 'J/P']]
print('-' * 60)
print(f'{"P2P (ours)":>20} {p2p_accs[0]:>8.4f} {p2p_accs[1]:>8.4f} {p2p_accs[2]:>8.4f} {p2p_accs[3]:>8.4f} {np.mean(p2p_accs):>8.4f}')
print(f'{"P2P (paper)":>20} {"0.9321":>8} {"0.9475":>8} {"0.9306":>8} {"0.8858":>8} {"0.9240":>8}')

In [ ]:
# 7.3 Confusion matrices
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Per-dimension confusion matrices
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, tc, pc) in zip(axes, dims):
    cm = confusion_matrix(valid[tc], valid[pc])
    ConfusionMatrixDisplay(cm, display_labels=name.split('/')).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name} (acc={results[name]["acc"]:.3f})')
plt.suptitle('P2P Per-Dimension Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'confusion_matrices.png', dpi=150)
plt.show()

# Full 16-way confusion matrix
print('\n16-way Classification Report:')
print(classification_report(valid['type'], valid['pred'], zero_division=0))

all_types = sorted(VALID_TYPES)
cm16 = confusion_matrix(valid['type'], valid['pred'], labels=all_types)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm16, annot=True, fmt='d', xticklabels=all_types, yticklabels=all_types, cmap='Blues', ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('16-way MBTI Confusion Matrix')
plt.tight_layout()
plt.savefig(ARTIFACTS / 'confusion_matrix_16way.png', dpi=150)
plt.show()